# 🥪 Jersey Mike's US Store Location Scraper

**Strat 411 Project**

This notebook scrapes all Jersey Mike's store locations in the United States and saves the data to both **CSV** and **JSON** files.

### Data Collected
| Field | Description |
|---|---|
| `location_name` | Name of the store location |
| `street_address` | Street address |
| `city` | City |
| `state` | Two-letter state abbreviation |
| `zip_code` | ZIP code |

### How to Use
1. Click **Runtime → Run all** (or press `Ctrl+F9`) to run every cell
2. Wait for the scraper to finish (progress is logged below each cell)
3. Output CSV and JSON files will be downloaded to your computer automatically

### Two Scraping Methods
- **API method** (default) — Uses Jersey Mike’s internal API. Faster and more reliable.
- **HTML method** (fallback) — Scrapes the website HTML with BeautifulSoup using CSS selectors. Use this if the API stops working.

Change the method in the **Configuration** section below.

---
## 1. Install Dependencies

Installs the Python libraries needed for web scraping. These are:
- **requests** — makes HTTP requests to fetch web pages and API data
- **beautifulsoup4** — parses HTML to extract data (used by the HTML method)
- **lxml** — fast HTML parser backend for BeautifulSoup

In [ ]:
!pip install requests beautifulsoup4 lxml -q

---
## 2. Imports

In [ ]:
import csv
import io
import json
import logging
import os
import sys
import time

import requests
from bs4 import BeautifulSoup

# Google Colab file download helper
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("All imports successful!")
print(f"Running in Google Colab: {IN_COLAB}")

---
## 3. Configuration

All settings for the scraper are defined here. You can change:
- **Scraping method**: `"api"` (recommended) or `"html"` (fallback)
- **Rate limiting**: Delay between requests to be respectful to the server
- **Output paths**: Where to save the CSV and JSON files
- **CSS selectors**: Used only by the HTML method (see Section 3b below)

In [ ]:
# ==============================================================================
# SCRAPING METHOD
# ==============================================================================
# Choose the scraping method:
#   "api"  - Use Jersey Mike's internal API (recommended, faster & more reliable)
#   "html" - Scrape the website HTML directly (uses CSS selectors below)
SCRAPE_METHOD = "api"

# ==============================================================================
# BASE URLS
# ==============================================================================

# Target URL: the main USA locations page
BASE_URL = "https://www.jerseymikes.com"
USA_LOCATIONS_URL = f"{BASE_URL}/locations/usa"

# Jersey Mike's internal API endpoints (used by the API method)
# These return structured JSON data for all store locations
API_BASE_URL = "https://bapi.prd.jerseymikes.com/api/v0"
API_SUBDIVISIONS_URL = f"{API_BASE_URL}/stores/subdivision"
API_STORES_BY_STATE_URL = f"{API_BASE_URL}/stores/bySubdivision"

# ==============================================================================
# RATE LIMITING
# ==============================================================================

# Delay (in seconds) between consecutive HTTP requests.
# Increase this if you get 429 (Too Many Requests) errors.
REQUEST_DELAY = 1.5

# Maximum number of retry attempts for failed requests
MAX_RETRIES = 3

# Delay (in seconds) between retry attempts (multiplied by attempt number)
RETRY_DELAY = 5

# Request timeout in seconds
REQUEST_TIMEOUT = 30

# ==============================================================================
# REQUEST HEADERS
# ==============================================================================

USER_AGENT = (
    "JerseyMikesLocationScraper/1.0 "
    "(Strat 411 School Project; educational use only)"
)

REQUEST_HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}

API_HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept": "application/json",
}

# ==============================================================================
# OUTPUT SETTINGS
# ==============================================================================

OUTPUT_DIR = "output"
CSV_FILENAME = "jersey_mikes_locations.csv"
JSON_FILENAME = "jersey_mikes_locations.json"

# ==============================================================================
# API SETTINGS
# ==============================================================================

API_PAGE_SIZE = 1000  # Stores to fetch per state per request

# ==============================================================================
# US STATES (used by the HTML method to iterate through state pages)
# ==============================================================================

US_STATES = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "DC", "FL",
    "GA", "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME",
    "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH",
    "NJ", "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI",
    "SC", "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI",
    "WY",
]

print("Configuration loaded!")

### 3b. CSS Selectors (HTML method only)

These selectors are used **only** by the HTML scraping method. If you switch
to `SCRAPE_METHOD = "html"` and it stops working, you may need to update
the selectors below.

#### How to Find / Update Selectors

1. Open https://www.jerseymikes.com/locations/usa in **Chrome** or **Firefox**
2. Click on any state to open its locations page (e.g., California)
3. **Right-click** on a store’s name or address → select **“Inspect”**
4. The DevTools panel will open showing the HTML for that element
5. Look at the HTML **tag**, **class names**, and **attributes** (especially `itemprop`)
6. Update the matching selector string below

Jersey Mike’s uses [Schema.org microdata](https://schema.org/PostalAddress)
for addresses. The typical HTML structure looks like:

```html
<div class="location-card">
  <h3 itemprop="name">Store Name</h3>
  <p itemprop="address">
    <span itemprop="streetAddress">123 Main St</span>
    <span itemprop="addressLocality">Anytown</span>,
    <span itemprop="addressRegion">CA</span>
    <span itemprop="postalCode">90210</span>
  </p>
</div>
```

#### Quick Selector Reference

| Selector | Example | What it matches |
|---|---|---|
| By tag | `h3` | All `<h3>` elements |
| By class | `.location-card` | Elements with `class="location-card"` |
| By attribute | `[itemprop='name']` | Elements with `itemprop="name"` |
| Combined | `div.card h3` | `<h3>` inside `<div class="card">` |

In [ ]:
# CSS SELECTORS for the HTML scraping method.
# Each key describes what the selector targets on the page.
# Update these if the Jersey Mike's website HTML structure changes.

SELECTORS = {
    # Links to individual state pages found on /locations/usa
    # Inspect a state link (e.g. "Alabama") to find the pattern
    "state_links": "a[href*='/locations/']",

    # The container wrapping each store's address (Schema.org microdata)
    # Inspect the address block of any store to find this element
    "address_container": "[itemprop='address']",

    # Store name element
    # Inspect the store name heading/link to find this element
    "store_name": "[itemprop='name']",

    # Street address line
    # Inspect the street address text to find this element
    "street_address": "[itemprop='streetAddress']",

    # City name
    # Inspect the city text to find this element
    "city": "[itemprop='addressLocality']",

    # State abbreviation
    # Inspect the state text to find this element
    "state": "[itemprop='addressRegion']",

    # ZIP code
    # Inspect the zip code text to find this element
    "zip_code": "[itemprop='postalCode']",

    # Pagination links at the bottom of state pages
    # Inspect the page number links to find this element
    "pagination": "ul.pagination a",
}

print("CSS selectors loaded!")

---
## 4. Setup Logging & Helpers

In [ ]:
# Configure logging so progress is printed below each cell
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
logger = logging.getLogger("scraper")


def make_request(url, headers=None, params=None):
    """
    Make an HTTP GET request with retry logic and rate limiting.

    Args:
        url: The URL to request.
        headers: Optional dict of HTTP headers.
        params: Optional dict of query parameters.

    Returns:
        requests.Response object on success.

    Raises:
        requests.RequestException: If all retries fail.
    """
    if headers is None:
        headers = REQUEST_HEADERS

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            logger.debug("Requesting: %s (attempt %d/%d)", url, attempt, MAX_RETRIES)
            response = requests.get(
                url, headers=headers, params=params, timeout=REQUEST_TIMEOUT,
            )
            response.raise_for_status()

            # Rate limiting: wait between requests
            time.sleep(REQUEST_DELAY)
            return response

        except requests.RequestException as e:
            logger.warning(
                "Request failed (attempt %d/%d): %s - %s",
                attempt, MAX_RETRIES, url, e,
            )
            if attempt < MAX_RETRIES:
                wait_time = RETRY_DELAY * attempt
                logger.info("Retrying in %d seconds...", wait_time)
                time.sleep(wait_time)
            else:
                logger.error("All %d attempts failed for: %s", MAX_RETRIES, url)
                raise


print("Logging and helpers ready!")

---
## 5. Scraper Functions

### 5a. API Method (Recommended)

Uses Jersey Mike’s internal API to fetch structured JSON data.

**How it works:**
1. Calls the subdivisions endpoint to get a list of all US states
2. For each state, calls the stores endpoint to get all locations
3. Extracts name, address, city, state, and zip from the JSON response

In [ ]:
def scrape_via_api():
    """
    Scrape store locations using Jersey Mike's internal API.

    Returns:
        list of dict: Each dict has keys location_name, street_address,
                      city, state, zip_code.
    """
    locations = []

    # Step 1: Get all US state/subdivision codes
    logger.info("Fetching US state subdivision codes from API...")
    try:
        response = make_request(API_SUBDIVISIONS_URL, headers=API_HEADERS)
        data = response.json()
        subdivisions = data.get("data", {}).get("storeSubdivisionCounts", [])
        logger.info("Found %d state subdivisions", len(subdivisions))
    except (requests.RequestException, KeyError, json.JSONDecodeError) as e:
        logger.error("Failed to fetch subdivision data: %s", e)
        return locations

    # Step 2: For each state, fetch all store locations
    for i, subdivision in enumerate(subdivisions, 1):
        state_code = subdivision.get("subdivisionCode", "")
        store_count = subdivision.get("count", 0)
        logger.info(
            "Fetching stores for %s (%d expected) [%d/%d states]",
            state_code, store_count, i, len(subdivisions),
        )

        try:
            response = make_request(
                API_STORES_BY_STATE_URL,
                headers=API_HEADERS,
                params={
                    "subdivisionCode": state_code,
                    "pageSize": API_PAGE_SIZE,
                    "pageNumber": 0,
                },
            )
            data = response.json()
            stores = data.get("data", {}).get("openStores", [])

            for store in stores:
                address = store.get("address", {})
                location = {
                    "location_name": store.get("name", "").strip(),
                    "street_address": address.get("street1", "").strip(),
                    "city": address.get("city", "").strip(),
                    "state": address.get("state", "").strip(),
                    "zip_code": address.get("zip", "").strip(),
                }

                # Append street2 if present (e.g., suite number)
                street2 = address.get("street2", "").strip()
                if street2:
                    location["street_address"] += f", {street2}"

                locations.append(location)

            logger.info("  -> Retrieved %d stores for %s", len(stores), state_code)

        except (requests.RequestException, KeyError, json.JSONDecodeError) as e:
            logger.error("Failed to fetch stores for %s: %s", state_code, e)
            continue

    return locations


print("API scraper function defined!")

### 5b. HTML Method (Fallback)

Scrapes the website HTML using BeautifulSoup and the CSS selectors defined
in the Configuration section above.

**How it works:**
1. Visits each US state’s location page (e.g., `/locations/CA`)
2. Determines how many pages of results exist (pagination)
3. On each page, finds location cards using the CSS selectors
4. Extracts name, address, city, state, and zip from the HTML

In [ ]:
def get_max_pages(soup):
    """
    Determine the max number of pages for a state's location listing.

    Inspects pagination links at the bottom of the page.

    Args:
        soup: BeautifulSoup object of the page.

    Returns:
        int: The highest page number found (at least 1).
    """
    max_page = 1
    pagination_links = soup.select(SELECTORS["pagination"])

    for link in pagination_links:
        text = link.get_text(strip=True)
        if text.isdigit():
            page_num = int(text)
            if page_num > max_page:
                max_page = page_num

    return max_page


def parse_location_card(card, state_code):
    """
    Parse a single location card element to extract store info.

    Uses the CSS selectors from the SELECTORS dict.

    Args:
        card: BeautifulSoup element for a store location.
        state_code: The state abbreviation being scraped.

    Returns:
        dict or None: Location data, or None if parsing fails.
    """
    try:
        name_el = card.select_one(SELECTORS["store_name"])
        street_el = card.select_one(SELECTORS["street_address"])
        city_el = card.select_one(SELECTORS["city"])
        state_el = card.select_one(SELECTORS["state"])
        zip_el = card.select_one(SELECTORS["zip_code"])

        location = {
            "location_name": name_el.get_text(strip=True) if name_el else "",
            "street_address": street_el.get_text(strip=True) if street_el else "",
            "city": city_el.get_text(strip=True) if city_el else "",
            "state": state_el.get_text(strip=True) if state_el else state_code,
            "zip_code": zip_el.get_text(strip=True) if zip_el else "",
        }

        # Clean up trailing commas from city names
        location["city"] = location["city"].rstrip(",").strip()

        if location["street_address"]:
            return location
        return None

    except Exception as e:
        logger.debug("Error parsing location card: %s", e)
        return None


def scrape_state_html(state_code):
    """
    Scrape all store locations for a single US state via HTML.

    Args:
        state_code: Two-letter state abbreviation (e.g., 'CA').

    Returns:
        list of dict: Locations found in this state.
    """
    locations = []
    state_url = f"{BASE_URL}/locations/{state_code}"

    try:
        response = make_request(f"{state_url}?page=1")
        soup = BeautifulSoup(response.text, "lxml")
        max_page = get_max_pages(soup)
        logger.info("  State %s has %d page(s)", state_code, max_page)
    except requests.RequestException:
        logger.error("  Failed to load initial page for %s", state_code)
        return locations

    for page_num in range(1, max_page + 1):
        try:
            if page_num > 1:
                response = make_request(f"{state_url}?page={page_num}")
                soup = BeautifulSoup(response.text, "lxml")

            address_containers = soup.select(SELECTORS["address_container"])

            if not address_containers:
                logger.warning(
                    "  No containers found on page %d. "
                    "CSS selectors may need updating.",
                    page_num,
                )
                continue

            for container in address_containers:
                card = container.parent
                location = parse_location_card(card, state_code)
                if location:
                    locations.append(location)

        except requests.RequestException:
            logger.error("  Failed page %d for %s", page_num, state_code)
            continue

    return locations


def scrape_via_html():
    """
    Scrape store locations by parsing HTML pages for each US state.

    Returns:
        list of dict: All store locations found.
    """
    all_locations = []
    logger.info("Starting HTML scraping for %d states...", len(US_STATES))

    for i, state_code in enumerate(US_STATES, 1):
        logger.info("Scraping state %s [%d/%d]", state_code, i, len(US_STATES))
        state_locations = scrape_state_html(state_code)
        all_locations.extend(state_locations)
        logger.info("  -> Found %d locations in %s", len(state_locations), state_code)

    return all_locations


print("HTML scraper functions defined!")

---
## 6. Run the Scraper 🚀

This cell runs the scraper using the method selected in the Configuration
section. Progress is logged below.

In [ ]:
logger.info("=" * 60)
logger.info("Jersey Mike's US Store Location Scraper")
logger.info("=" * 60)
logger.info("Method: %s", SCRAPE_METHOD)
logger.info("Rate limit delay: %s seconds", REQUEST_DELAY)
logger.info("=" * 60)

start_time = time.time()

if SCRAPE_METHOD == "api":
    logger.info("Using API method (recommended)...")
    locations = scrape_via_api()
else:
    logger.info("Using HTML scraping method...")
    logger.info(
        "NOTE: If selectors don't match, update them in the Configuration section."
    )
    locations = scrape_via_html()

elapsed = time.time() - start_time

logger.info("=" * 60)
logger.info("Scraping complete!")
logger.info("Total locations found: %d", len(locations))
logger.info("Time elapsed: %.1f seconds", elapsed)
logger.info("=" * 60)

---
## 7. Preview the Data

Shows a preview of the first 10 locations scraped.

In [ ]:
if locations:
    # Try to use pandas for a nice table display (pre-installed in Colab)
    try:
        import pandas as pd
        df = pd.DataFrame(locations)
        print(f"\nTotal locations: {len(df)}")
        print(f"States represented: {df['state'].nunique()}")
        print(f"\nLocations per state (top 10):")
        print(df['state'].value_counts().head(10).to_string())
        print(f"\nFirst 10 locations:")
        display(df.head(10))
    except ImportError:
        print(f"\nTotal locations: {len(locations)}")
        print("\nFirst 10 locations:")
        for loc in locations[:10]:
            print(f"  {loc['location_name']} - {loc['street_address']}, "
                  f"{loc['city']}, {loc['state']} {loc['zip_code']}")
else:
    print("No locations found! Check the logs above for errors.")
    if SCRAPE_METHOD == "html":
        print("TIP: The CSS selectors may need updating. See Section 3b.")

---
## 8. Save & Download Output Files

Saves the scraped data to CSV and JSON files, then downloads them
to your computer automatically (in Google Colab).

In [ ]:
if not locations:
    print("No locations to save. Please re-run the scraper (Section 6).")
else:
    # Create output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    csv_path = os.path.join(OUTPUT_DIR, CSV_FILENAME)
    json_path = os.path.join(OUTPUT_DIR, JSON_FILENAME)

    # --- Save CSV ---
    fieldnames = ["location_name", "street_address", "city", "state", "zip_code"]
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(locations)
    print(f"Saved {len(locations)} locations to CSV: {csv_path}")

    # --- Save JSON ---
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(locations, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(locations)} locations to JSON: {json_path}")

    # --- Download files (Google Colab only) ---
    if IN_COLAB:
        print("\nDownloading files to your computer...")
        colab_files.download(csv_path)
        colab_files.download(json_path)
        print("Downloads started! Check your browser's download bar.")
    else:
        print(f"\nFiles saved to: {os.path.abspath(OUTPUT_DIR)}/")

---
## 9. (Optional) Save to Google Drive

Run this cell if you want to save the output files directly to your
Google Drive instead of (or in addition to) downloading them.

In [ ]:
# Uncomment the lines below to save to Google Drive:

# from google.colab import drive
# drive.mount('/content/drive')
#
# drive_output_dir = '/content/drive/My Drive/Strat 411/jersey_mikes_data'
# os.makedirs(drive_output_dir, exist_ok=True)
#
# import shutil
# shutil.copy(csv_path, os.path.join(drive_output_dir, CSV_FILENAME))
# shutil.copy(json_path, os.path.join(drive_output_dir, JSON_FILENAME))
# print(f"Files copied to Google Drive: {drive_output_dir}")